In [38]:
import os

In [39]:
%pwd

'C:\\Users\\DELL\\Documents\\Complete-ml-project\\Complete-ml-deployment'

In [40]:
os.chdir(r"C:\Users\DELL\Documents\Complete-ml-project\Complete-ml-deployment")

In [41]:
%pwd

'C:\\Users\\DELL\\Documents\\Complete-ml-project\\Complete-ml-deployment'

In [42]:

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict

In [43]:
from mlproject.constants import *
from mlproject.utils.common import read_yaml, create_directories

In [44]:

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir = config.unzip_data_dir,
            all_schema=schema,
        )

        return data_validation_config

In [46]:

import os
from mlproject import logger
import pandas as pd

In [ ]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    
    def validate_all_columns(self)-> bool:
        try:
            validation_status = None

            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)

            all_schema = self.config.all_schema.keys()

            
            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"column Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"column Validation status: {validation_status}")

            return validation_status
        
        except Exception as e:
            raise e


    def validate_data_types(self) -> bool:
        """Check if each column has the correct datatype as per schema."""
        try:
            data = pd.read_csv(self.config.unzip_data_dir)
            validation_status = True

            for col, expected_dtype in self.config.all_schema.items():
                if col in data.columns:
                    actual_dtype = str(data[col].dtype)
                    if expected_dtype not in actual_dtype:
                        validation_status = False
                        with open(self.config.STATUS_FILE, 'a') as f:
                            f.write(f"Datatype mismatch in column '{col}': expected {expected_dtype}, got {actual_dtype}\n")

            with open(self.config.STATUS_FILE, 'a') as f:
                f.write(f"Data Type Validation Status: {validation_status}\n")

            return validation_status

        except Exception as e:
            raise e

In [48]:
try:
    # Initialize config
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()

    # Initialize Data Validation
    data_validation = DataValidation(config=data_validation_config)

    # Run validation steps
    columns_valid = data_validation.validate_all_columns()
    dtypes_valid = data_validation.validate_data_types()
except Exception as e:
    raise e

[2025-10-19 22:15:24,559: INFO: common]: yaml file: config\config.yaml loaded successfully
[2025-10-19 22:15:24,573: INFO: common]: yaml file: params.yaml loaded successfully
[2025-10-19 22:15:24,590: INFO: common]: yaml file: schema.yaml loaded successfully
[2025-10-19 22:15:24,593: INFO: common]: created directory at: artifacts
[2025-10-19 22:15:24,595: INFO: common]: created directory at: artifacts/data_validation
